<a href="https://colab.research.google.com/github/kananshr13/CSET-343-Ai-in-Healthcare/blob/main/Lab_Experiment_7_CSET_343.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikeras -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasClassifier

np.random.seed(42)

In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
        "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]

df = pd.read_csv(url, names=cols)
# If you have the Kaggle file instead: df = pd.read_csv("diabetes.csv")

print(df.shape)
df.head()

(768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
print(df.info())
print(df.describe())
print("\nClass distribution:\n", df["Outcome"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std    

In [ ]:
zero_invalid = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_invalid] = df[zero_invalid].replace(0, np.nan)
df[zero_invalid] = df[zero_invalid].fillna(df[zero_invalid].median())

print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [ ]:
X = df.drop("Outcome", axis=1).values
y = df["Outcome"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train.shape, X_test.shape)

(614, 8) (154, 8)


In [ ]:
import tensorflow as tf
from sklearn.base import BaseEstimator, ClassifierMixin

class KerasWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, neurons=12, activation="relu", init="glorot_uniform",
                 dropout_rate=0.0, learning_rate=0.001, epochs=50, batch_size=16):
        self.neurons = neurons
        self.activation = activation
        self.init = init
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

    def _build(self, n_features):
        model = Sequential([
            Input(shape=(n_features,)),
            Dense(self.neurons, activation=self.activation, kernel_initializer=self.init),
            Dropout(self.dropout_rate),
            Dense(8, activation=self.activation, kernel_initializer=self.init),
            Dense(1, activation="sigmoid", kernel_initializer=self.init),
        ])
        model.compile(loss="binary_crossentropy",
                      optimizer=Adam(learning_rate=self.learning_rate),
                      metrics=["accuracy"])
        return model

    def fit(self, X, y):
        tf.keras.backend.clear_session()
        self.classes_ = np.unique(y)
        self.model_ = self._build(X.shape[1])
        self.model_.fit(X, y, epochs=self.epochs, batch_size=self.batch_size, verbose=0)
        return self

    def predict(self, X):
        proba = self.model_.predict(X, verbose=0).ravel()
        return (proba > 0.5).astype(int)

In [ ]:
def run_grid(param_grid, **fixed):
    grid = GridSearchCV(estimator=KerasWrapper(**fixed), param_grid=param_grid,
                        cv=3, n_jobs=1, scoring="accuracy")
    result = grid.fit(X_train, y_train)
    print(f"Best: {result.best_score_:.4f} using {result.best_params_}\n")
    for m, s, p in zip(result.cv_results_["mean_test_score"],
                       result.cv_results_["std_test_score"],
                       result.cv_results_["params"]):
        print(f"{m:.4f} ({s:.4f}) with: {p}")
    return result

best = {}

In [ ]:
res = run_grid({"batch_size": [10, 20, 40, 60], "epochs": [10, 50, 100]})
best.update(res.best_params_)

Best: 0.7883 using {'batch_size': 10, 'epochs': 50}

0.7720 (0.0148) with: {'batch_size': 10, 'epochs': 10}
0.7883 (0.0263) with: {'batch_size': 10, 'epochs': 50}
0.7623 (0.0316) with: {'batch_size': 10, 'epochs': 100}
0.7476 (0.0216) with: {'batch_size': 20, 'epochs': 10}
0.7688 (0.0395) with: {'batch_size': 20, 'epochs': 50}
0.7607 (0.0416) with: {'batch_size': 20, 'epochs': 100}
0.6970 (0.0247) with: {'batch_size': 40, 'epochs': 10}
0.7639 (0.0041) with: {'batch_size': 40, 'epochs': 50}
0.7785 (0.0125) with: {'batch_size': 40, 'epochs': 100}
0.7134 (0.0365) with: {'batch_size': 60, 'epochs': 10}
0.7769 (0.0136) with: {'batch_size': 60, 'epochs': 50}
0.7590 (0.0089) with: {'batch_size': 60, 'epochs': 100}


In [ ]:
res = run_grid({"learning_rate": [0.0001, 0.001, 0.01, 0.1]}, **best)
best.update(res.best_params_)

Best: 0.7623 using {'learning_rate': 0.001}

0.7492 (0.0235) with: {'learning_rate': 0.0001}
0.7623 (0.0290) with: {'learning_rate': 0.001}
0.7183 (0.0134) with: {'learning_rate': 0.01}
0.7036 (0.0190) with: {'learning_rate': 0.1}


In [ ]:
res = run_grid({"init": ["random_uniform", "lecun_uniform", "random_normal", "zeros",
                         "glorot_normal", "glorot_uniform", "he_normal", "he_uniform"]},
               **best)
best.update(res.best_params_)

Best: 0.7834 using {'init': 'lecun_uniform'}

0.7769 (0.0262) with: {'init': 'random_uniform'}
0.7834 (0.0179) with: {'init': 'lecun_uniform'}
0.7818 (0.0279) with: {'init': 'random_normal'}
0.6515 (0.0020) with: {'init': 'zeros'}
0.7736 (0.0194) with: {'init': 'glorot_normal'}
0.7639 (0.0203) with: {'init': 'glorot_uniform'}
0.7639 (0.0222) with: {'init': 'he_normal'}
0.7704 (0.0235) with: {'init': 'he_uniform'}


In [ ]:
res = run_grid({"activation": ["softmax", "softplus", "softsign", "relu",
                               "tanh", "sigmoid", "hard_sigmoid", "linear"]},
               **best)
best.update(res.best_params_)

Best: 0.7818 using {'activation': 'linear'}

0.7769 (0.0118) with: {'activation': 'softmax'}
0.7753 (0.0179) with: {'activation': 'softplus'}
0.7802 (0.0297) with: {'activation': 'softsign'}
0.7655 (0.0221) with: {'activation': 'relu'}
0.7769 (0.0185) with: {'activation': 'tanh'}
0.7687 (0.0096) with: {'activation': 'sigmoid'}
0.7736 (0.0096) with: {'activation': 'hard_sigmoid'}
0.7818 (0.0192) with: {'activation': 'linear'}


In [16]:
res = run_grid({"dropout_rate": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]}, **best)
best.update(res.best_params_)

Best: 0.7818 using {'dropout_rate': 0.0}

0.7818 (0.0192) with: {'dropout_rate': 0.0}
0.7802 (0.0203) with: {'dropout_rate': 0.1}
0.7802 (0.0169) with: {'dropout_rate': 0.2}
0.7802 (0.0169) with: {'dropout_rate': 0.3}
0.7785 (0.0146) with: {'dropout_rate': 0.4}
0.7736 (0.0157) with: {'dropout_rate': 0.5}


In [17]:
res = run_grid({"neurons": [1, 5, 10, 15, 20, 25, 30]}, **best)
best.update(res.best_params_)
print("\nFinal tuned hyperparameters:", best)

Best: 0.7851 using {'neurons': 5}

0.7704 (0.0115) with: {'neurons': 1}
0.7851 (0.0169) with: {'neurons': 5}
0.7834 (0.0180) with: {'neurons': 10}
0.7818 (0.0192) with: {'neurons': 15}
0.7834 (0.0180) with: {'neurons': 20}
0.7818 (0.0192) with: {'neurons': 25}
0.7818 (0.0192) with: {'neurons': 30}

Final tuned hyperparameters: {'batch_size': 10, 'epochs': 50, 'learning_rate': 0.001, 'init': 'lecun_uniform', 'activation': 'linear', 'dropout_rate': 0.0, 'neurons': 5}


In [18]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

final_clf = KerasWrapper(**best).fit(X_train, y_train)
y_pred = final_clf.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Test Accuracy: 0.7012987012987013

Confusion Matrix:
 [[82 18]
 [28 26]]

Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.82      0.78       100
           1       0.59      0.48      0.53        54

    accuracy                           0.70       154
   macro avg       0.67      0.65      0.66       154
weighted avg       0.69      0.70      0.69       154

